# pig-math TTS Phase 1A — 試聽多個 voice

**目標**：用 `edge-tts` 把 ~20 句典型語音用 4 個候選 voice 各生成一份，下載 zip 試聽，挑出最適合 pig-math 的兒童音色。

## 用法

1. 在 Colab 開啟此 notebook（不需 GPU，CPU runtime 即可）
2. Runtime → Run all
3. 最後一格會把 `voice-samples.zip` 下載到本機
4. 解壓後依資料夾結構聽（`voice-samples/<voice>/<phrase_id>.mp3`），挑喜歡的 voice 回報

## Voice 候選

| Voice ID | 性別 | 風格 | 預期適配 |
|---|---|---|---|
| `zh-TW-HsiaoYuNeural` | 女 | 年輕、活潑 | 像姐姐／幼教老師 ⭐推薦 |
| `zh-TW-HsiaoChenNeural` | 女 | 成熟、溫暖 | 像媽媽／阿姨 |
| `zh-TW-YunJheNeural` | 男 | 沉穩 | 像爸爸／哥哥 |
| `zh-CN-XiaoyiNeural` | 女 | 甜美、年輕 | 對照組（中國腔，可能不適合） |

## Step 1：安裝 edge-tts

In [1]:
!pip install -q edge-tts

## Step 2：定義測試語句

從 pig-math 既有 codebase 抽出代表性句子（數字、題目、回饋、休息提示）。

In [2]:
# 測試句：涵蓋 pig-math 各種語音場景
TEST_PHRASES = [
    # 數字（單獨念）
    ("num_5",       "五"),
    ("num_21",      "二十一"),
    ("num_100",     "一百"),

    # 題目（最高頻場景）
    ("problem_add",      "五 加 三，等於多少？"),
    ("problem_sub",      "十二 減 七，等於多少？"),
    ("problem_mul",      "四 乘以 六，等於多少？"),
    ("problem_div",      "八十四 除以 四，等於多少？"),
    ("problem_compare",  "二十一 和 三十二，哪個比較大？"),
    ("problem_seq",      "第三題，二 加 二，等於多少？"),

    # 答對 / 答錯回饋
    ("feedback_correct1", "答對了！你好棒！"),
    ("feedback_correct2", "哇，全對耶！"),
    ("feedback_wrong",    "差一點點，再試試看"),
    ("feedback_encourage","你已經很努力了，繼續加油"),

    # 中場休息（沿用 BreakReminder 既有句）
    ("break_water",      "辛苦囉，休息一下，記得喝口水喔"),
    ("break_stretch",    "做得很棒，站起來動一動"),
    ("break_eyes",       "讓眼睛休息一下，看看遠遠的地方"),

    # 描紅引導
    ("trace_3",          "我們來練習寫三吧"),
    ("trace_intro",      "沿著虛線描寫看看"),

    # 學習模式 / 教學提示
    ("learn_hint",       "湊十法：五加五等於十，然後再加三，就是十三"),
    ("learn_intro",      "我們來學位值，數字裡每一位都有名字"),
]

VOICES = [
    "zh-TW-HsiaoYuNeural",
    "zh-TW-HsiaoChenNeural",
    "zh-TW-YunJheNeural",
    "zh-CN-XiaoyiNeural",
]

print(f'會生成 {len(TEST_PHRASES) * len(VOICES)} 個 MP3')

會生成 80 個 MP3


## Step 3：生成 MP3

rate=-10% 微調語速給小孩聽起來舒服一點。每個 voice 一個資料夾。

In [3]:
import asyncio
import os
import edge_tts

OUTPUT_DIR = "voice-samples"
os.makedirs(OUTPUT_DIR, exist_ok=True)

async def generate(voice: str, phrase_id: str, text: str):
    voice_dir = os.path.join(OUTPUT_DIR, voice.replace('Neural', ''))
    os.makedirs(voice_dir, exist_ok=True)
    out_path = os.path.join(voice_dir, f"{phrase_id}.mp3")
    # rate=-10% 比預設稍慢，給小孩聽更舒服；pitch 不調（neural voice 自帶語調）
    communicate = edge_tts.Communicate(text, voice, rate="-10%")
    await communicate.save(out_path)
    return out_path

async def main():
    tasks = []
    for voice in VOICES:
        for phrase_id, text in TEST_PHRASES:
            tasks.append(generate(voice, phrase_id, text))
    # 並行生成
    paths = await asyncio.gather(*tasks)
    print(f'✓ 生成 {len(paths)} 個檔案')
    # 顯示每個 voice 的檔案數
    for voice in VOICES:
        voice_dir = os.path.join(OUTPUT_DIR, voice.replace('Neural', ''))
        count = len(os.listdir(voice_dir))
        print(f'  {voice}: {count} 檔')

await main()

✓ 生成 80 個檔案
  zh-TW-HsiaoYuNeural: 20 檔
  zh-TW-HsiaoChenNeural: 20 檔
  zh-TW-YunJheNeural: 20 檔
  zh-CN-XiaoyiNeural: 20 檔


## Step 4：產生試聽索引（HTML）

解壓 zip 後直接開 `index.html` 可以在瀏覽器一鍵聽完所有句子，方便比較。

In [4]:
html = ['<!DOCTYPE html><html><head><meta charset="utf-8"><title>pig-math TTS 試聽</title>']
html.append('<style>body{font-family:system-ui;padding:20px;max-width:1200px;margin:auto;}')
html.append('table{border-collapse:collapse;width:100%;}td,th{border:1px solid #ddd;padding:10px;text-align:left;}')
html.append('th{background:#f5f5f5;}audio{width:200px;}.phrase{max-width:300px;}</style></head><body>')
html.append('<h1>pig-math TTS 試聽（Phase 1A）</h1>')
html.append('<p>聽完後挑最喜歡的 voice 回報。語速統一 -10%。</p>')
html.append('<table><thead><tr><th>場景 / 句子</th>')
for v in VOICES:
    html.append(f'<th>{v.replace("Neural", "")}</th>')
html.append('</tr></thead><tbody>')
for phrase_id, text in TEST_PHRASES:
    html.append(f'<tr><td class="phrase"><code>{phrase_id}</code><br>{text}</td>')
    for v in VOICES:
        v_dir = v.replace('Neural', '')
        html.append(f'<td><audio controls preload="none" src="{v_dir}/{phrase_id}.mp3"></audio></td>')
    html.append('</tr>')
html.append('</tbody></table></body></html>')

with open(os.path.join(OUTPUT_DIR, 'index.html'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(html))
print('✓ 試聽頁產生：voice-samples/index.html')

✓ 試聽頁產生：voice-samples/index.html


## Step 5：打包 + 下載 zip

In [5]:
import shutil
zip_path = shutil.make_archive('voice-samples', 'zip', '.', OUTPUT_DIR)
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'✓ 打包：{zip_path}（{size_mb:.1f} MB）')

# Colab 環境自動下載
try:
    from google.colab import files
    files.download(zip_path)
    print('✓ 開始下載 voice-samples.zip')
except ImportError:
    print('（非 Colab 環境，請手動下載 voice-samples.zip）')

✓ 打包：/content/voice-samples.zip（1.3 MB）


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ 開始下載 voice-samples.zip


In [7]:
# Step 5.5：把 zip 上傳到臨時 file host 拿一個 URL，可靠
#
# 為什麼這樣做：VS Code 的 Colab 擴充功能不吃 files.download 的 JS 回呼，
# 所以 cell 10 的 "files.download" 不會真的下載到本機。換一個方式：
# 把 zip 上傳到 0x0.st（24h 自動刪除），印出 URL，你貼到瀏覽器就能下載。

import os
import subprocess

abs_zip = os.path.abspath('voice-samples.zip')
size_mb = os.path.getsize(abs_zip) / 1024 / 1024
print(f'zip 位置: {abs_zip}（{size_mb:.1f} MB）')
print()

# 嘗試 0x0.st（最穩定的 24h temp file host，無需 auth）
def upload_to_0x0(path):
    r = subprocess.run(
        ['curl', '-s', '-F', f'file=@{path}', '-F', 'expires=1', 'https://0x0.st'],
        capture_output=True, text=True, timeout=60,
    )
    if r.returncode == 0 and r.stdout.strip().startswith('http'):
        return r.stdout.strip()
    return None

# 備案：file.io
def upload_to_fileio(path):
    r = subprocess.run(
        ['curl', '-s', '-F', f'file=@{path}', 'https://file.io/?expires=1d'],
        capture_output=True, text=True, timeout=60,
    )
    if r.returncode != 0:
        return None
    try:
        import json
        data = json.loads(r.stdout)
        if data.get('success') and data.get('link'):
            return data['link']
    except Exception:
        pass
    return None

print('▶ 上傳到 0x0.st ...')
url = upload_to_0x0(abs_zip)
if not url:
    print('  0x0.st 失敗，試 file.io ...')
    url = upload_to_fileio(abs_zip)

if url:
    print()
    print('═══════════════════════════════════════════════════════════')
    print(f'  ⬇  下載連結（24h 內有效）:')
    print(f'      {url}')
    print('═══════════════════════════════════════════════════════════')
    print()
    print('▼ 在本機終端機跑（會自動下載 + 解壓到 Math/tmp/）')
    print('  cd ~/Downloads/pig-math-tmp')
    print(f'  curl -sLO {url}')
    print(f'  mv $(basename {url}) voice-samples.zip')
    print('  unzip -o voice-samples.zip')
    print()
    print('  接著開 voice-samples/index.html 試聽')
else:
    print('✗ 兩個 file host 都上傳失敗')
    print('  退路：Colab 介面左側「檔案」面板 → 找 voice-samples.zip → 右鍵 Download')

zip 位置: /content/voice-samples.zip（1.3 MB）

▶ 上傳到 0x0.st ...
  0x0.st 失敗，試 file.io ...
✗ 兩個 file host 都上傳失敗
  退路：Colab 介面左側「檔案」面板 → 找 voice-samples.zip → 右鍵 Download


## 下一步

1. 解壓 `voice-samples.zip`
2. 在電腦上打開 `voice-samples/index.html`，從瀏覽器一鍵聽完所有 voice × 句子
3. 回報最喜歡的 voice（例如「HsiaoYu 最像姐姐，但講題目時 HsiaoChen 比較穩」這種具體 feedback）
4. 我會用你選的 voice 跑 Phase 1B 全量生成（~300 句）